# Jupyter Chat Components Demo

This notebook showcases the components provided by the
[`jupyter-chat-components`](https://github.com/jtpio/jupyter-chat-components) extension.

The extension registers a custom MIME renderer for the `application/vnd.jupyter.chat.components`
MIME type, which can be used to display rich, interactive UI components in notebook outputs.

## Programmatic Usage

Components are rendered via the Jupyter display protocol using the custom MIME type.

The display protocol uses two parts:

- **Data bundle**: `{MIME_TYPE: "<component-name>"}` — selects which component to render
- **Metadata**: `{MIME_TYPE: {...}}` — provides the component's props/configuration

This notebook uses the [JavaScript kernel](https://github.com/jupyterlite/javascript-kernel) and its
[`display()`](https://github.com/jupyterlite/javascript-kernel#rich-output) function
with the `_toMime()` method to send custom MIME bundles:

```javascript
const MIME_TYPE = "application/vnd.jupyter.chat.components";

display(
  // Data bundle: selects the component to render
  { _toMime: () => ({ [MIME_TYPE]: "tool-call" }) },
  // Metadata: provides the component's configuration
  {
    [MIME_TYPE]: {
      toolName: "my_tool",
      input: '{"key": "value"}',
      status: "completed",
    },
  },
);
```

## ToolCall Component

The `ToolCall` component renders tool execution information in a collapsible UI element,
showing the tool name, input, output, and status.

**Component name**: `"tool-call"`

### Metadata Fields

| Field | Type | Required | Description |
|-------|------|----------|-------------|
| `toolName` | `string` | Yes | Name of the tool being called |
| `input` | `string` | Yes | Input data or parameters passed to the tool |
| `status` | `ToolCallStatus` | Yes | Current execution status (see below) |
| `summary` | `string` | No | Short summary displayed next to the tool name |
| `output` | `string` | No | Output or result from the tool execution |

### Status Types

| Status | Description |
|--------|-------------|
| `"pending"` | Tool is currently running |
| `"awaiting_approval"` | Tool requires user approval before executing |
| `"approved"` | User approved execution and the tool is running |
| `"rejected"` | User rejected execution |
| `"completed"` | Tool finished successfully |
| `"error"` | Tool encountered an error |

### Examples

The following helper function wraps `display()` for convenience:

In [ ]:
const MIME_TYPE = "application/vnd.jupyter.chat.components";

function showToolCall({ toolName, input, status = "completed", summary, output }) {
  display(
    { _toMime: () => ({ [MIME_TYPE]: "tool-call" }) },
    { [MIME_TYPE]: { toolName, input, status, summary, output } },
  );
}

#### Completed Tool Call

A tool call that has finished executing successfully:

In [ ]:
showToolCall({
  toolName: "list_files",
  input: '{"path": "/home/user/project"}',
  status: "completed",
  summary: "Listed 5 files",
  output: '["README.md", "setup.py", "main.py", "utils.py", "test_main.py"]',
})

#### Pending Tool Call

A tool call that is currently running:

In [ ]:
showToolCall({
  toolName: "execute_code",
  input: 'console.log("Hello, world!")',
  status: "pending",
  summary: "Running JavaScript code",
})

#### Error Tool Call

A tool call that encountered an error:

In [ ]:
showToolCall({
  toolName: "read_file",
  input: '{"path": "/nonexistent/file.txt"}',
  status: "error",
  output: "Error: ENOENT: no such file or directory, open '/nonexistent/file.txt'",
})

#### Awaiting Approval

A tool call that requires user approval before executing:

In [ ]:
showToolCall({
  toolName: "delete_file",
  input: '{"path": "/home/user/important_data.csv"}',
  status: "awaiting_approval",
  summary: "Delete a file",
})

#### All Statuses

Here are all the supported tool call statuses displayed together:

In [ ]:
const statuses = ["pending", "awaiting_approval", "approved", "rejected", "completed", "error"];

for (const status of statuses) {
  showToolCall({
    toolName: "example_tool",
    input: JSON.stringify({ action: "demo", status }),
    status,
    summary: `Status: ${status}`,
    ...(["completed", "error", "rejected"].includes(status) && { output: "Some output" }),
  });
}

## Grouped Tool Calls Component

The `GroupedToolCalls` component renders grouped tool call rows with expandable details,
success and failure states,
inline diffs, and permission options.

**Component name**: `"grouped-tool-calls"`

### Metadata Fields

| Field | Type | Required | Description |
|-------|------|----------|-------------|
| `toolCalls` | `Array` | Yes | Ordered list of tool call entries |

Each entry can include `toolCallId`, `title`, `kind`, `status`, `rawInput`,
`rawOutput`, `locations`, `diffs`, `permissionOptions`, `permissionStatus`,
`selectedOptionId`, and `sessionId`.


### Examples

The helper below renders the grouped tool call component.

In [ ]:
function showGroupedToolCalls(metadata) {
  display(
    { _toMime: () => ({ [MIME_TYPE]: "grouped-tool-calls" }) },
    { [MIME_TYPE]: metadata },
  );
}

#### Mixed Tool Call States

A grouped example showing completed, in-progress, failed, and resolved edit tool calls:

In [ ]:
showGroupedToolCalls({
  toolCalls: [
    {
      toolCallId: "tc-1",
      title: "Read README.md",
      kind: "read",
      status: "completed",
      locations: ["/home/user/project/README.md"],
    },
    {
      toolCallId: "tc-2",
      title: "Search src for tool call renderers",
      kind: "search",
      status: "in_progress",
    },
    {
      toolCallId: "tc-3",
      title: "Run lint",
      kind: "execute",
      status: "failed",
      rawOutput: "npm ERR! Missing dependency: eslint",
    },
    {
      toolCallId: "tc-4",
      title: "Edit src/components/index.ts",
      kind: "edit",
      status: "completed",
      permissionStatus: "resolved",
      selectedOptionId: "allow-once",
      permissionOptions: [
        { optionId: "allow-once", name: "Allow once", kind: "allow_once" },
        { optionId: "reject-once", name: "Reject", kind: "reject_once" },
      ],
      diffs: [
        {
          path: "/home/user/project/src/components/index.ts",
          oldText: "export * from './tool-call';",
          newText: "export * from './tool-call';\nexport * from './grouped-tool-calls';",
        },
      ],
    },
  ],
})

#### Approval Flow With Diffs

The approval buttons are rendered here but disabled because this standalone demo notebook
does not register a permission callback with the renderer factory.

In [ ]:
showGroupedToolCalls({
  toolCalls: [
    {
      toolCallId: "tc-5",
      title: "Edit src/components/grouped-tool-calls.tsx",
      kind: "edit",
      permissionStatus: "pending",
      sessionId: "demo-session",
      permissionOptions: [
        { optionId: "allow-once", name: "Allow once", kind: "allow_once" },
        { optionId: "allow-always", name: "Always allow", kind: "allow_always" },
        { optionId: "reject-once", name: "Reject", kind: "reject_once" },
      ],
      diffs: [
        {
          path: "/home/user/project/src/components/grouped-tool-calls.tsx",
          oldText: "return null;",
          newText: "return <div className=\"jp-ai-tool-calls\">...</div>;",
        },
      ],
    },
    {
      toolCallId: "tc-6",
      title: "Running: rm build/output.log...",
      kind: "execute",
      permissionStatus: "pending",
      sessionId: "demo-session",
      permissionOptions: [
        { optionId: "allow-once", name: "Allow once", kind: "allow_once" },
        { optionId: "reject-once", name: "Reject", kind: "reject_once" },
      ],
      rawInput: {
        command: "rm build/output.log",
        cwd: "/home/user/project",
      },
    },
  ],
})

## InlineDiff Component

The `InlineDiff` component renders one or more file or notebook cell diffs directly in chat output.

**Component name**: `"inline-diff"`

### Metadata Fields

| Field | Type | Required | Description |
|-------|------|----------|-------------|
| `diffs` | `Array` | Yes | List of file diffs to render |

Each diff item uses the following shape:

| Field | Type | Required | Description |
|-------|------|----------|-------------|
| `target` | `object` | Yes | Structured file or cell target |
| `label` | `string` | No | Explicit header label override |
| `newText` | `string` | Yes | Updated file contents |
| `oldText` | `string` | No | Previous file contents |

Each diff item must include a `target`.

Supported targets include:

- File: `{ kind: "file", path: "src/example.py" }`
- Cell: `{ kind: "cell", notebookPath: "notebooks/analysis.ipynb", cellIndex: 11 }`

Notebook cell labels are rendered from the notebook name plus the 1-based cell number.


In [ ]:
function showInlineDiff(diff) {
  display(
    { _toMime: () => ({ [MIME_TYPE]: "inline-diff" }) },
    { [MIME_TYPE]: { diffs: [diff] } },
  );
}

### Example

Define the original and updated file contents in variables, then render their diff:

In [ ]:
const oldText = `def greet(name):
    return f\"Hello {name}\"`;

const newText = `def greet(name, excited = false):
    suffix = \"!\" if excited else \"\"\n    return f\"Hello {name}{suffix}\"`;

In [ ]:
showInlineDiff({
  target: { kind: "file", path: "src/example.py" },
  oldText,
  newText,
});

### Notebook Cell Example

Structured targets can describe a specific notebook cell source diff:

In [ ]:
const notebookOldText = `total = revenue.sum()`;

const notebookNewText = `total = revenue.sum()\navg = revenue.mean()`;

showInlineDiff({
  target: {
    kind: "cell",
    notebookPath: "notebooks/analysis.ipynb",
    cellIndex: 11,
  },
  oldText: notebookOldText,
  newText: notebookNewText,
});

### Longer Diff with Truncation

When a diff exceeds 20 lines, the component truncates the output and shows a
clickable "... N more lines" button to expand the remaining lines:

In [ ]:
const longOldText = `import os
import sys


def read_file(path):
    """Read a file and return its contents."""
    with open(path, "r") as f:
        return f.read()


def write_file(path, content):
    """Write content to a file."""
    with open(path, "w") as f:
        f.write(content)


def list_files(directory):
    """List all files in a directory."""
    return os.listdir(directory)


def get_extension(filename):
    """Get the file extension."""
    return os.path.splitext(filename)[1]


def is_python_file(filename):
    """Check if a file is a Python file."""
    return get_extension(filename) == ".py"


def count_lines(path):
    """Count the number of lines in a file."""
    content = read_file(path)
    return len(content.splitlines())`;

const longNewText = `import os
import sys
from pathlib import Path


def read_file(path):
    """Read a file and return its contents."""
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


def write_file(path, content):
    """Write content to a file."""
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)


def list_files(directory, pattern="*"):
    """List all files in a directory matching a pattern."""
    return [str(p) for p in Path(directory).glob(pattern)]


def get_extension(filename):
    """Get the file extension."""
    return Path(filename).suffix


def is_python_file(filename):
    """Check if a file is a Python file."""
    return get_extension(filename) == ".py"


def count_lines(path):
    """Count the number of lines in a file."""
    content = read_file(path)
    return len(content.splitlines())


def find_files(directory, extension):
    """Find all files with a given extension."""
    return [f for f in list_files(directory) if f.endswith(extension)]`;

In [ ]:
showInlineDiff({
  target: { kind: "file", path: "src/utils.py" },
  oldText: longOldText,
  newText: longNewText,
});

## MessageQueue Component

The `MessageQueue` component displays pending messages as bubbles in the chat to
represent messages that are queued while the agent is busy.

**Component name**: `"message-queue"`

### Metadata Fields

| Field | Type | Required | Description |
|-------|------|----------|-------------|
| `messages` | `Array` | Yes | List of queued messages to display |
| `targetId` | `string` | No | Target ID used when removing a queued message |

Each message follows this schema:

| Field | Type | Required | Description |
|-------|------|----------|-------------|
| `id` | `string` | Yes | Unique identifier for the message |
| `body` | `string` | Yes | Text content of the queued message |

### Note on `removeQueuedMessage`

The component accepts a `removeQueuedMessage(targetId, messageId)` callback, which renders
a ✕ button on each message bubble. It is injected by the [`jupyterlite/ai`](https://github.com/jupyterlite/ai/pull/302) components factory.

### Examples

In [3]:
function showMessageQueue(messages, targetId) {
  display(
    { _toMime: () => ({ [MIME_TYPE]: "message-queue" }) },
    { [MIME_TYPE]: { messages, ...(targetId !== undefined ? { targetId } : {}) } },
  );
}

#### Single Queued Message

In [ ]:
showMessageQueue([
  { id: "msg-1", body: "Create a simple linear regression notebook" },
]);

#### Multiple Queued Messages

In [ ]:
showMessageQueue([
  { id: "msg-1", body: "What is the time complexity of quicksort?" },
  { id: "msg-2", body: "Can you show me an example implementation?" },
  { id: "msg-3", body: "Compare it to merge sort" },
]);